# Milestone 2 - Neural Network Models

This notebook keeps the modeling simple and focuses on NN baselines:
1. sklearn MLP (`MLPRegressor`)
2. PyTorch MLP (optional, runs only if `torch` is installed)

Targets:
- `review_scores_rating` (RQ2)
- `price_num` using `log1p` transform (RQ3)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print("torch available:", TORCH_AVAILABLE)

In [ ]:
DATA_PATH = "../data/raw/listings.csv"

COMMON_NUMERIC = [
    "accommodates", "bedrooms", "bathrooms", "beds",
    "minimum_nights", "availability_365", "number_of_reviews"
]

COMMON_CATEGORICAL = [
    "room_type", "property_type", "neighbourhood_cleansed",
    "host_is_superhost", "instant_bookable"
]

RATING_FEATURES = COMMON_NUMERIC + COMMON_CATEGORICAL + ["price_num"]
PRICE_FEATURES = COMMON_NUMERIC + COMMON_CATEGORICAL + ["review_scores_rating"]


def parse_price(series):
    return pd.to_numeric(
        series.astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False),
        errors="coerce",
    )


def normalize_tf(value):
    if pd.isna(value):
        return "Unknown"
    t = str(value).strip().lower()
    if t in {"t", "true", "1", "yes", "y"}:
        return "t"
    if t in {"f", "false", "0", "no", "n"}:
        return "f"
    return "Unknown"


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(features):
    num_cols = [c for c in features if c in COMMON_NUMERIC or c in {"price_num", "review_scores_rating"}]
    cat_cols = [c for c in features if c in COMMON_CATEGORICAL]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_ohe())
        ]), cat_cols),
    ])


df = pd.read_csv(DATA_PATH, low_memory=False)
df["price_num"] = parse_price(df["price"])
for col in COMMON_NUMERIC + ["review_scores_rating", "price_num"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["host_is_superhost", "instant_bookable"]:
    df[col] = df[col].map(normalize_tf)
for col in COMMON_CATEGORICAL:
    df[col] = df[col].fillna("Unknown").astype(str)

print("rows:", len(df))

In [ ]:
def run_sklearn_mlp(target, features, use_log1p=False):
    frame = df[features + [target]].dropna(subset=[target]).copy()
    X, y = frame[features], frame[target].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    pre = make_preprocessor(features)
    model = MLPRegressor(
        hidden_layer_sizes=(128, 64),
        max_iter=350,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42,
    )
    pipe = Pipeline([("pre", pre), ("model", model)])

    y_fit = np.log1p(y_train) if use_log1p else y_train
    pipe.fit(X_train, y_fit)

    pred = pipe.predict(X_test)
    if use_log1p:
        pred = np.maximum(np.expm1(pred), 0.0)
    else:
        pred = np.clip(pred, 1.0, 5.0)

    return {
        "model": pipe,
        "mae": float(mean_absolute_error(y_test, pred)),
        "r2": float(r2_score(y_test, pred)),
    }


rating_sklearn = run_sklearn_mlp("review_scores_rating", RATING_FEATURES, use_log1p=False)
price_sklearn = run_sklearn_mlp("price_num", PRICE_FEATURES, use_log1p=True)

print("sklearn rating | MAE:", round(rating_sklearn["mae"], 3), "| R2:", round(rating_sklearn["r2"], 3))
print("sklearn price  | MAE:", round(price_sklearn["mae"], 2), "| R2:", round(price_sklearn["r2"], 3))

In [ ]:
if TORCH_AVAILABLE:
    class TorchRegressor(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Dropout(0.15),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 1),
            )

        def forward(self, x):
            return self.net(x)


    def run_torch_mlp(target, features, use_log1p=False):
        frame = df[features + [target]].dropna(subset=[target]).copy()
        X, y = frame[features], frame[target].to_numpy()
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pre = make_preprocessor(features)
        X_train_t = np.asarray(pre.fit_transform(X_train), dtype=np.float32)
        X_test_t = np.asarray(pre.transform(X_test), dtype=np.float32)

        y_train_fit = np.log1p(y_train) if use_log1p else y_train

        model = TorchRegressor(X_train_t.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        loss_fn = nn.SmoothL1Loss()

        data = TensorDataset(
            torch.tensor(X_train_t, dtype=torch.float32),
            torch.tensor(y_train_fit.reshape(-1, 1), dtype=torch.float32),
        )
        loader = DataLoader(data, batch_size=128, shuffle=True)

        model.train()
        for _ in range(80):
            for xb, yb in loader:
                optimizer.zero_grad()
                pred = model(xb)
                loss = loss_fn(pred, yb)
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(torch.tensor(X_test_t, dtype=torch.float32)).numpy().reshape(-1)

        if use_log1p:
            pred = np.maximum(np.expm1(pred), 0.0)
        else:
            pred = np.clip(pred, 1.0, 5.0)

        return {
            "mae": float(mean_absolute_error(y_test, pred)),
            "r2": float(r2_score(y_test, pred)),
        }


    rating_torch = run_torch_mlp("review_scores_rating", RATING_FEATURES, use_log1p=False)
    price_torch = run_torch_mlp("price_num", PRICE_FEATURES, use_log1p=True)

    print("torch rating   | MAE:", round(rating_torch["mae"], 3), "| R2:", round(rating_torch["r2"], 3))
    print("torch price    | MAE:", round(price_torch["mae"], 2), "| R2:", round(price_torch["r2"], 3))
else:
    print("Torch not installed. Skipping torch model block.")